<a href="https://colab.research.google.com/github/SohailVibeCoder/IB9AU---GenAI/blob/main/Task_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Name**: Sohail Essajee (5757504)


The task showed that TabPFN slightly outperformed both Random Forest and XGBoost overall, achieving the highest accuracy (0.8223 vs. 0.8155 and 0.8127) and the strongest precision for the default class. However, all three models had very similar weaknesses, with low recall for actual defaulters (around 0.35–0.36), meaning many positive default cases were still missed. This suggests that while TabPFN is a very strong out-of-the-box tabular model, the bigger issue in this problem is handling class imbalance and improving detection of the minority default class.

A key insight was that all three models performed much better at predicting non-defaults than defaults, with low recall for the default class across the board. This showed that the main challenge in the task was class imbalance and correctly identifying defaulters, rather than just maximising overall accuracy.

In [ ]:
import kagglehub
import pandas as pd
import os

dataset_path = kagglehub.dataset_download("uciml/default-of-credit-card-clients-dataset")

# List contents of the downloaded dataset directory to find the data file
files_in_dataset = os.listdir(dataset_path)
print(f"Files in the dataset directory: {files_in_dataset}")

# Assuming the main data file is a CSV and is directly in the downloaded path
# You might need to adjust the filename if it's different or in a subdirectory
# For this dataset, it's typically 'UCI_Credit_Card.csv'

# Construct the full path to the CSV file
csv_file_name = 'UCI_Credit_Card.csv'
full_csv_path = os.path.join(dataset_path, csv_file_name)

# Load the CSV into a pandas DataFrame
df = pd.read_csv(full_csv_path)

print("Data loaded into DataFrame 'df' successfully.")
print(df.head())

In [ ]:
df

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from tabpfn_client import TabPFNClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Prepare Data
# Remove ID and separate target
X = df.drop(['ID', 'default.payment.next.month'], axis=1)
y = df['default.payment.next.month']

# Split data (75% train, 25% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Scale data (important for many TabPFN implementations)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Note: TabPFN is optimized for smaller datasets (<10k samples).
# If the dataset is large, we sub-sample for the TabPFN training step
# to ensure it fits within memory limits.
X_train_tab = X_train_scaled[:5000]
y_train_tab = y_train[:5000]

# 2. Initialize Models
rf = RandomForestClassifier(n_estimators=100, random_state=42)
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
tabpfn = TabPFNClassifier(random_state=42) # Removed device='cpu'

# 3. Training and Prediction
models = {'Random Forest': rf, 'XGBoost': xgb, 'TabPFN': tabpfn}
results = {}

for name, model in models.items():
    if name == 'TabPFN':
        # Convert y_train_tab to a 1D array if it's a Series or DataFrame column
        if isinstance(y_train_tab, (pd.Series, pd.DataFrame)):
            y_train_tab_fit = y_train_tab.values.ravel()
        else:
            y_train_tab_fit = y_train_tab
        model.fit(X_train_tab, y_train_tab_fit)
    else:
        # Convert y_train to a 1D array if it's a Series or DataFrame column
        if isinstance(y_train, (pd.Series, pd.DataFrame)):
            y_train_fit = y_train.values.ravel()
        else:
            y_train_fit = y_train
        model.fit(X_train_scaled, y_train_fit)

    preds = model.predict(X_test_scaled)
    results[name] = {
        'Accuracy': accuracy_score(y_test, preds),
        'Report': classification_report(y_test, preds)
    }

# 4. Output Comparison
for name, metrics in results.items():
    print(f"--- {name} ---")
    print(f"Accuracy: {metrics['Accuracy']:.4f}")
    print(metrics['Report'])

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [15:02:06] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
Processing: 100%|██████████| [00:02<00:00]

--- Random Forest ---
Accuracy: 0.8155
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      5873
           1       0.63      0.36      0.46      1627

    accuracy                           0.82      7500
   macro avg       0.74      0.65      0.67      7500
weighted avg       0.80      0.82      0.80      7500

--- XGBoost ---
Accuracy: 0.8127
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      5873
           1       0.62      0.36      0.46      1627

    accuracy                           0.81      7500
   macro avg       0.73      0.65      0.67      7500
weighted avg       0.79      0.81      0.79      7500

--- TabPFN ---
Accuracy: 0.8223
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      5873
           1       0.67      0.35      0.46      1627

    accuracy                           0.82      7500
   macro avg       0.76 